# Import Modules

In [ ]:
from utils import *
import os
import joblib

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
MODEL_DIR = "model"

# Load Datasets

In [ ]:
df_statlog, scaler_statlog = get_preprocessed("statlog", "statlog")
df_chd, scaler_chd = get_preprocessed("chd", "chd")
df_framingham, scaler_framingham = get_preprocessed("framingham", "framingham")
df_heart, scaler_heart = get_preprocessed("heart", "heart")
df_stroke, scaler_stroke = get_preprocessed("stroke", "stroke")

# Training

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0)
}

In [ ]:
def split_xy(df: pd.DataFrame, target_col: str = "target"):
    if target_col is None or target_col not in df.columns:
        raise ValueError("Cannot find target column")

    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y

In [ ]:
def evaluate_and_fit(name: str, estimator, X, y):
    cv_res = cross_validate(
        estimator, X, y,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
    )

    summary = {k: (np.mean(v), np.std(v)) for k, v in cv_res.items() if k.startswith("test_")}
    for metric, (m, s) in summary.items():
        print(f"{name:>12} - {metric.replace('test_', ''):>9}: {m:.4f} ± {s:.4f}")

    estimator.fit(X, y)

    model_path = os.path.join(MODEL_DIR, f"{name}.joblib")
    joblib.dump(estimator, model_path)

    return summary

In [ ]:
def train_dataset(dataset_name: str, df: pd.DataFrame, scaler: MinMaxScaler, target_col: str = "target"):
    X, y = split_xy(df, target_col=target_col)

    scaler_path = os.path.join(MODEL_DIR, f"{dataset_name}_scaler.joblib")
    joblib.dump(scaler, scaler_path)

    models = {
        "logreg": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "knn": KNeighborsClassifier(n_neighbors=5),
        "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "svm": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
        "naive_bayes": GaussianNB(),
        "random_forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    }

    results = {}
    for model_name, est in models.items():
        full_name = f"{dataset_name}_{model_name}"
        results[full_name] = evaluate_and_fit(full_name, est, X, y)

    return results

In [ ]:
all_results = {}
all_results.update(train_dataset("statlog", df_statlog, scaler_statlog))
all_results.update(train_dataset("chd", df_chd, scaler_chd))
all_results.update(train_dataset("framingham", df_framingham, scaler_framingham))
all_results.update(train_dataset("heart", df_heart, scaler_heart))
all_results.update(train_dataset("stroke", df_stroke, scaler_stroke))

In [ ]:
all_results_df = pd.DataFrame({
    "dataset": [k.split("_")[0] for k in all_results.keys()],
    "model": [k.split("_")[1] for k in all_results.keys()],
    'accuracy_mean': [v['test_accuracy'][0] for v in all_results.values()],
    'accuracy_std': [v['test_accuracy'][1] for v in all_results.values()],
    'precision_mean': [v['test_precision'][0] for v in all_results.values()],
    'precision_std': [v['test_precision'][1] for v in all_results.values()],
    'recall_mean': [v['test_recall'][0] for v in all_results.values()],
    'recall_std': [v['test_recall'][1] for v in all_results.values()],
    "f1_mean": [v['test_f1'][0] for v in all_results.values()],
    "f1_std": [v['test_f1'][1] for v in all_results.values()],
})

all_results_df